# Limpieza de tablas auxiliares IGDB

En este notebook se limpian las tablas auxiliares descargadas desde IGDB:

- `igdb_generos.csv`
- `igdb_plataformas.csv`
- `igdb_involved_companies.csv`
- `igdb_empresas.csv`

El objetivo es:

- Estandarizar tipos de datos.
- Eliminar duplicados.
- Filtrar la información a los juegos que sobrevivieron la limpieza principal.
- Generar versiones limpias en `data/limpios/` para usar luego en el EDA y el modelo.


In [ ]:
from pathlib import Path
import pandas as pd

# Rutas de datos crudos y carpeta de salida
ruta_no_limpios = Path("..") / ".." / "data" / "raw"
ruta_limpios = Path("..") / ".." / "data" / "processed"
ruta_limpios.mkdir(parents=True, exist_ok=True)

# Cargar juegos limpios para filtrar las auxiliares
df_juegos_limpio = pd.read_csv(ruta_limpios / "juegos_igdb_limpio.csv")
ids_juegos_validos = df_juegos_limpio["id"].unique().tolist()

df_juegos_limpio.shape


(186076, 17)

## 1. Limpieza de géneros (`igdb_generos.csv`)

En esta tabla cada fila representa un género de juego.  
Los pasos son:

- Cargar el CSV original.
- Asegurar que `id` sea entero.
- Eliminar posibles filas duplicadas por `id`.
- Guardar la versión limpia.


In [2]:
df_generos = pd.read_csv(ruta_no_limpios / "igdb_generos.csv").copy()

# Asegurar tipo entero para id
df_generos["id"] = pd.to_numeric(df_generos["id"], errors="coerce").astype("Int64")

# Eliminar duplicados por id
df_generos = df_generos.drop_duplicates(subset="id", keep="first")

df_generos.info()
df_generos.head()

df_generos.to_csv(ruta_limpios / "igdb_generos_limpio.csv", index=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      23 non-null     Int64 
 1   name    23 non-null     object
 2   slug    23 non-null     object
dtypes: Int64(1), object(2)
memory usage: 707.0+ bytes


## 2. Limpieza de plataformas (`igdb_plataformas.csv`)

En esta tabla cada fila es una plataforma (PC, consola, móvil, etc.).

Pasos:

- Cargar el CSV original.
- Asegurar que `id` sea entero.
- Quitar duplicados por `id`.
- Guardar la versión limpia.


In [3]:
df_plataformas = pd.read_csv(ruta_no_limpios / "igdb_plataformas.csv").copy()

# Asegurar tipo entero para id
df_plataformas["id"] = pd.to_numeric(df_plataformas["id"], errors="coerce").astype("Int64")

# Eliminar duplicados por id
df_plataformas = df_plataformas.drop_duplicates(subset="id", keep="first")

df_plataformas.info()
df_plataformas.head()

df_plataformas.to_csv(ruta_limpios / "igdb_plataformas_limpio.csv", index=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               220 non-null    Int64  
 1   name             220 non-null    object 
 2   slug             220 non-null    object 
 3   platform_family  55 non-null     float64
dtypes: Int64(1), float64(1), object(2)
memory usage: 7.2+ KB


## 3. Limpieza de relaciones juego–empresa (`igdb_involved_companies.csv`)

Esta tabla relaciona juegos con empresas que participan en su desarrollo o publicación.
Cada fila indica si una empresa fue developer, publisher, etc. de un juego.

Pasos:

- Cargar el CSV original.
- Filtrar solo las filas cuyo `game` está en el conjunto de juegos limpios.
- Arreglar tipos de columnas (`id`, `game`, `company` como enteros, flags booleanos).
- Eliminar duplicados por `id`.
- Guardar la versión limpia.


In [4]:
df_involved = pd.read_csv(ruta_no_limpios / "igdb_involved_companies.csv").copy()

# Filtrar a juegos válidos (que quedaron en juegos_igdb_limpio)
df_involved = df_involved[df_involved["game"].isin(ids_juegos_validos)].copy()

# Tipos enteros para ids
for col in ["id", "game", "company"]:
    df_involved[col] = pd.to_numeric(df_involved[col], errors="coerce").astype("Int64")

# Flags booleanos
for col in ["developer", "publisher", "porting", "supporting"]:
    if col in df_involved.columns:
        df_involved[col] = df_involved[col].fillna(False).astype(bool)

# Eliminar duplicados por id
df_involved = df_involved.drop_duplicates(subset="id", keep="first")

df_involved.info()
df_involved.head()

df_involved.to_csv(ruta_limpios / "igdb_involved_companies_limpio.csv", index=False)


<class 'pandas.core.frame.DataFrame'>
Index: 147240 entries, 1199 to 186090
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   id          147240 non-null  Int64
 1   company     147240 non-null  Int64
 2   developer   147240 non-null  bool 
 3   game        147240 non-null  Int64
 4   porting     147240 non-null  bool 
 5   publisher   147240 non-null  bool 
 6   supporting  147240 non-null  bool 
dtypes: Int64(3), bool(4)
memory usage: 5.5 MB


## 4. Limpieza de empresas (`igdb_empresas.csv`)

Esta tabla describe compañías (desarrolladoras, publishers, etc.).

Pasos:

- Cargar el CSV original.
- Estandarizar tipos de datos (`id`, `country`).
- Convertir `start_date` a fecha legible y extraer año de inicio.
- Eliminar duplicados por `id`.
- Guardar la versión limpia.


In [5]:
df_empresas = pd.read_csv(ruta_no_limpios / "igdb_empresas.csv").copy()

# id como entero
df_empresas["id"] = pd.to_numeric(df_empresas["id"], errors="coerce").astype("Int64")

# país como entero con missing permitido
df_empresas["country"] = pd.to_numeric(df_empresas["country"], errors="coerce").astype("Int64")

# Convertir start_date (timestamp) a fecha y año de inicio
df_empresas["fecha_inicio"] = pd.to_datetime(
    df_empresas["start_date"],
    unit="s",
    errors="coerce"
)
df_empresas["año_inicio"] = df_empresas["fecha_inicio"].dt.year.astype("Int64")

# Eliminar duplicados por id
df_empresas = df_empresas.drop_duplicates(subset="id", keep="first")

df_empresas.info()
df_empresas.head()

df_empresas.to_csv(ruta_limpios / "igdb_empresas_limpio.csv", index=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1130 entries, 0 to 1129
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            1130 non-null   Int64         
 1   country       741 non-null    Int64         
 2   name          1130 non-null   object        
 3   slug          1130 non-null   object        
 4   start_date    623 non-null    float64       
 5   parent        276 non-null    float64       
 6   fecha_inicio  623 non-null    datetime64[ns]
 7   año_inicio    623 non-null    Int64         
dtypes: Int64(3), datetime64[ns](1), float64(2), object(2)
memory usage: 74.1+ KB


## 5. Resumen

A partir de los archivos originales en `data/no_limpios/` se generaron
las siguientes tablas auxiliares limpias en `data/limpios/`:

- `igdb_generos_limpio.csv`
- `igdb_plataformas_limpio.csv`
- `igdb_involved_companies_limpio.csv`
- `igdb_empresas_limpio.csv`

Estas tablas están listas para usarse en:

- el EDA (análisis por género, plataforma, tipo de estudio),
- la construcción de features (por ejemplo contar géneros, plataformas, clasificar estudios),
- y el posterior modelo de predicción de éxito de los juegos.
